# Bagré | 04: Atmospheric water demand
    **Goal:** calculate approximate FAO-56 daily reference evapotranspiration (ET0) from ERA5-Land. ET0 refers to a well-watered hypothetical grass reference surface; it is not actual crop water use. The ET0 series represents the coarse reanalysis cell near the pilot centre, not individual fields.

## 1. Connect Drive and Earth Engine
    Use a CPU runtime. Run this cell after every runtime restart. You need a registered Earth Engine Google Cloud project (not your Gmail address). Set its project ID below. The Drive folder must be in your My Drive, or a shortcut with the same name.

In [ ]:
%pip -q install earthengine-api rasterio pyproj
import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Bagre_ET_Water_Stress')
assert ROOT.is_dir(), f'Open Drive and confirm this folder exists: {ROOT}'
for sub in ['data', 'outputs', 'figures', 'qgis']:
    (ROOT / sub).mkdir(exist_ok=True)
PROJECT = 'YOUR_EARTH_ENGINE_PROJECT_ID'  # CHANGE THIS
assert PROJECT != 'YOUR_EARTH_ENGINE_PROJECT_ID', 'Set your registered Cloud project ID.'
try:
    ee.Initialize(project=PROJECT)
except ee.EEException:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print('Earth Engine connection:', ee.Number(1).getInfo())

## 2. Study design
    Reuse the approximate Bagré pilot window from the earlier water project. It includes agricultural land and other land covers; it is not an official irrigation boundary. January–April 2021 is a single dry-season demonstration. WorldCover 2021 cropland is a proxy, not a field survey. All end dates are exclusive. Do not change dates or AOI halfway through the series.

In [ ]:
import calendar
from datetime import date, timedelta
AOI_BOUNDS = [-0.59, 11.41, -0.47, 11.52]  # west, south, east, north
AOI = ee.Geometry.Rectangle(AOI_BOUNDS, proj='EPSG:4326', geodesic=False)
CENTER = AOI.centroid(1)
CRS = 'EPSG:32630'
MONTHS = [(f'2021-{m:02d}-01', f'2021-{m+1:02d}-01') for m in range(1, 5)]
S2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(AOI)
L8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterBounds(AOI)
      .filter(ee.Filter.eq('PROCESSING_LEVEL', 'L2SP')))
MOD = ee.ImageCollection('MODIS/061/MOD16A2GF').filterBounds(AOI)
WAP = ee.ImageCollection('FAO/WAPOR/3/L1_AETI_D').filterBounds(AOI)
ERA = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR').filterBounds(AOI)
WC = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')

## 3. Extract daily meteorology
    Temperature and dewpoint are in K, radiation is J/m²/day, pressure Pa, wind components m/s and precipitation m. The vector magnitude of daily-mean wind components can underestimate the daily mean scalar wind speed, so computed ET0 is approximate. We use the FAO-56 reference-surface radiation formulation, not reanalysis actual-surface net radiation.

In [ ]:
bands = ['temperature_2m_min','temperature_2m_max','dewpoint_temperature_2m',
         'surface_solar_radiation_downwards_sum','surface_pressure',
         'u_component_of_wind_10m','v_component_of_wind_10m','total_precipitation_sum']
col = ERA.filterDate(MONTHS[0][0], MONTHS[-1][1]).select(bands)
def row(im):
    im = ee.Image(im)
    values = im.reduceRegion(ee.Reducer.first(), CENTER, 11132)
    return ee.Feature(None, values).set('date', im.date().format('YYYY-MM-dd'))
features = ee.FeatureCollection(col.toList(col.size()).map(row)).getInfo()['features']
weather = pd.DataFrame([x['properties'] for x in features]).sort_values('date')
assert len(weather) == 120 and weather.date.nunique() == 120
assert weather[bands].notna().all().all()
display(weather.head())

## 4. Compute the FAO-56 Penman–Monteith equation
    It balances available energy and the atmosphere's ability to remove water vapor. Delta is the slope of the saturation vapor pressure curve, gamma is the psychrometric constant, and VPD is vapor pressure deficit. We use zero daily soil heat flux (FAO-56 approximation), latitude and an approximate center elevation from SRTM, and convert 10 m wind to 2 m.

In [ ]:
z = float(ee.Image('USGS/SRTMGL1_003').reduceRegion(
    ee.Reducer.first(), CENTER, 30).get('elevation').getInfo())
def fao56(df, latitude, elevation):
    tmin = df.temperature_2m_min - 273.15
    tmax = df.temperature_2m_max - 273.15
    t = (tmin+tmax)/2
    td = df.dewpoint_temperature_2m - 273.15
    svp = lambda x: 0.6108*np.exp(17.27*x/(x+237.3))
    es = (svp(tmin)+svp(tmax))/2
    ea = svp(td)
    vpd = np.maximum(es-ea, 0)
    delta = 4098*svp(t)/(t+237.3)**2
    gamma = 0.000665*df.surface_pressure/1000
    u10 = np.hypot(df.u_component_of_wind_10m, df.v_component_of_wind_10m)
    u2 = u10*4.87/np.log(67.8*10-5.42)
    rs = np.maximum(df.surface_solar_radiation_downwards_sum/1e6, 0)
    j = pd.to_datetime(df.date).dt.dayofyear
    phi = np.deg2rad(latitude)
    dr = 1+0.033*np.cos(2*np.pi*j/365)
    dec = 0.409*np.sin(2*np.pi*j/365-1.39)
    ws = np.arccos(np.clip(-np.tan(phi)*np.tan(dec), -1, 1))
    ra = (24*60/np.pi)*0.0820*dr*(ws*np.sin(phi)*np.sin(dec)+np.cos(phi)*np.cos(dec)*np.sin(ws))
    rso = (0.75+2e-5*elevation)*ra
    rns = 0.77*rs
    cloud_factor = 1.35*np.clip(rs/rso, 0.3, 1)-0.35
    rnl = 4.903e-9*((tmax+273.16)**4+(tmin+273.16)**4)/2*(0.34-0.14*np.sqrt(ea))*cloud_factor
    rn = rns-rnl
    et0 = (0.408*delta*rn+gamma*900/(t+273)*u2*vpd)/(delta+gamma*(1+0.34*u2))
    return df.assign(tair_c=t, vpd_kpa=vpd, wind2_ms=u2, radiation_mj=rs,
                     precip_mm=np.maximum(df.total_precipitation_sum*1000,0),
                     et0_mm=np.maximum(et0,0))
daily = fao56(weather, (AOI_BOUNDS[1]+AOI_BOUNDS[3])/2, z)
assert np.isfinite(daily.et0_mm).all() and daily.et0_mm.between(0,25).all()
daily.to_csv(ROOT/'data'/'weather_daily.csv', index=False)
monthly = daily.assign(month=daily.date.str[:7]).groupby('month').agg(
    et0_mm=('et0_mm','sum'), precip_mm=('precip_mm','sum'),
    vpd_kpa=('vpd_kpa','mean'), days=('et0_mm','size')).reset_index()
monthly.to_csv(ROOT/'data'/'weather_monthly.csv', index=False)
display(monthly)

In [ ]:
ax = daily.set_index(pd.to_datetime(daily.date))[['et0_mm','precip_mm']].plot(figsize=(12,4))
ax.set_ylabel('mm/day')
ax.set_title('Atmospheric water demand and precipitation (ERA5-Land)')
ax.figure.tight_layout()
ax.figure.savefig(ROOT/'figures'/'weather_et0.png', dpi=160)
plt.show()

**Checkpoint:** examine ET0 and rainfall. ET0 is not ETa; actual evapotranspiration can be above ET0 for some crops and conditions. A high ET0 does not mean a crop is stressed. Reference: [FAO-56](https://www.fao.org/4/x0490e/x0490e00.htm), [ERA5-Land daily](https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_DAILY_AGGR).